In [3]:
import pandas as pd
import numpy as np

from itertools import product

In [9]:
# TODO implement a brute force approach from the largest -> smallest submatrix
# return a list of compete submatrices
# construct the sets of indices (2^n -1)(2^m -1)
# order them on the area and then on the no of rows for each area

In [4]:
#TODO row protection by lambda in row_threshold?
def adaptive_prune_to_complete(
    data: pd.DataFrame,
    max_iter:int = 100,
    learning_rate:float = 0.001,
    shrink_coeff:float = 10,
    verbose: bool = True
):
    n_rows, n_cols = data.shape
    X = data.copy()
    # iteration count
    i = 1
    while(i <= max_iter):
        if verbose:
                print(
                    f"i={i}, shape={X.shape},\n"
                )
        
        if n_rows == 0 or n_cols == 0:
            return None
            
        # remove columns

        # pick threshold
        # fraction of elements in the least populated row
        # it doesn't matter if there's many, one is chosen
        col_avail = X.notna().sum(axis=0)
        min_fraction_col = (col_avail/n_rows).min()
        # fraction of missing + shrunk learning_rate
        col_threshold = min_fraction_col + (1 - min_fraction_col)/(shrink_coeff * i)

        if verbose:
            print(
                f"A column with min values has {col_avail.min():.3f} entries.\n"
                f"Threshold of column elimination is {col_threshold:.3f}, it eliminates columns with less than {int(col_threshold * n_rows)} entries."
            )
        
        # prune columns
        X = X.loc[:, col_avail >= col_threshold * n_rows]

        # update column count
        n_cols = X.shape[1]
        if n_cols == 0: 
            return None

        # remove rows
        
        # pick threshold
        # fraction of elements in the least populated row
        # it doesn't matter if there's many, one is chosen
        row_avail = X.notna().sum(axis=1)
        min_fraction_row = (row_avail/n_cols).min()
        # fraction of missing + shrunk learning_rate
        row_threshold = min_fraction_row + (1 - min_fraction_row)/(shrink_coeff * i)

        if verbose:
            print(
                f"A row with min values has {row_avail.min():.3f} entries.\n"
                f"Threshold of row elimination is {row_threshold:.3f}, it eliminates rows with less than {int(row_threshold * n_cols)} entries."
            )
        
        # prune rows
        X = X.loc[row_avail >= row_threshold * n_cols, :]

        # update row count
        n_rows = X.shape[0]
        if n_rows == 0:
            return None

        missing = X.isna().sum().sum()
        total = X.shape[0] * X.shape[1]

        if verbose:
            print(
                f"missing={missing}/{total}\n"
            )

        if missing == 0: break
        
        i += 1

    if i <= max_iter: return X
    else:
        print(f"{max_iter} iterations reached without convergence.")

In [5]:
rng = np.random.default_rng()

In [6]:
#TODO
# missingness only outside the completness indices
# tryout to incorporate the missingness and gain control
missingness = 0.2
to_nan = rng.choice(list(product(np.setdiff1d(np.arange(1, n_data_points_big + 1), rows), np.setdiff1d(np.arange(1, n_variables_big + 1), cols))), 
           int((n_data_points_big * n_variables_big)*missingness))

NameError: name 'n_data_points_big' is not defined

In [7]:
#TODO
# add metrics
# add controlled missingness

# compare rows first to colymns first cutting

best_area = []

T = 100
for t in np.arange(T):
    n_data_points = 9
    n_variables = 17
    complete = rng.standard_normal(size=(n_data_points, n_variables))
    
    n_data_points_big = 15
    n_variables_big = 18
    
    big = rng.standard_normal((n_data_points_big, n_variables_big))
    big[rng.integers(low=0, high=1, endpoint=True, size=(n_data_points_big, n_variables_big)) == 0] = np.nan
    
    rows = np.sort(rng.choice(n_data_points_big, size=n_data_points, replace=False))
    cols = np.sort(rng.choice(n_variables_big, size=n_variables, replace=False))
    
    big[np.ix_(rows,cols)] = complete
    
    df = pd.DataFrame(big)

    lrs = np.logspace(-3, 0, 10)
    shs = np.arange(2, 101, 10)
    
    results = {}
    
    for lr, sh in list(product(lrs, shs)):
        #print(lr, sh)
        results[lr, sh] = adaptive_prune_to_complete(df, max_iter=1000, learning_rate=lr, shrink_coeff=sh, verbose=False)
        
        assert results[lr, sh] is None or results[lr, sh].isna().sum().sum() == 0, f"{lr}, {sh}"

    summary = pd.DataFrame(
    [
        {
            "learning_rate": lr,
            "shrink_coeff": sh,
            "n_rows": result.shape[0],
            "n_cols": result.shape[1],
            "area": result.shape[0] * result.shape[1],
        }
        for (lr, sh), result in results.items()
        if result is not None
    ]
    )

    summary.sort_values("area", ascending=False, inplace = True)
    best_area.append(summary.iloc[0]["area"])
    top = summary[summary["area"] == best_area[t]]
    
    for lr, sh in top[["learning_rate", "shrink_coeff"]].itertuples(index=False):
        # print(lr, sh)
        
        best_matrix = results[lr, sh]
        # print(len(best_matrix.index), len(best_matrix.columns))
        
        # print(np.array_equal(best_matrix, complete))
    
        recovered_rows = best_matrix.index.intersection(rows)
        recovered_cols = best_matrix.columns.intersection(cols)
        recovered_area = len(recovered_rows) * len(recovered_cols)
    
        # print(recovered_area/(complete.shape[0] * complete.shape[1]))

np.mean(np.array(best_area))

np.float64(105.7)

In [344]:
n_data_points = 5
n_variables = 3
complete = rng.standard_normal(size=(n_data_points, n_variables))

n_data_points_big = 15
n_variables_big = 18

big = rng.standard_normal((n_data_points_big, n_variables_big))
big[rng.integers(low=0, high=1, endpoint=True, size=(n_data_points_big, n_variables_big)) == 0] = np.nan

rows = np.sort(rng.choice(n_data_points_big, size=n_data_points, replace=False))
cols = np.sort(rng.choice(n_variables_big, size=n_variables, replace=False))

big[np.ix_(rows,cols)] = complete

df = pd.DataFrame(big)

In [345]:
lrs = np.logspace(-3, 0, 10)
shs = np.arange(2, 101, 10)

results = {}

for lr, sh in list(product(lrs, shs)):
    #print(lr, sh)
    results[lr, sh] = adaptive_prune_to_complete(df, max_iter=1000, learning_rate=lr, shrink_coeff=sh, verbose=False)
    
    assert results[lr, sh] is None or results[lr, sh].isna().sum().sum() == 0, f"{lr}, {sh}"

In [346]:
summary = pd.DataFrame(
    [
        {
            "learning_rate": lr,
            "shrink_coeff": sh,
            "n_rows": result.shape[0],
            "n_cols": result.shape[1],
            "area": result.shape[0] * result.shape[1],
        }
        for (lr, sh), result in results.items()
        if result is not None
    ]
)

summary.sort_values("area", ascending=False, inplace = True)
best_area = summary.iloc[0]["area"]
top = summary[summary["area"] == best_area]

for lr, sh in top[["learning_rate", "shrink_coeff"]].itertuples(index=False):
    print(lr, sh)
    
    best_matrix = results[lr, sh]
    print(len(best_matrix.index), len(best_matrix.columns))
    
    print(np.array_equal(best_matrix, complete))

    recovered_rows = best_matrix.index.intersection(rows)
    recovered_cols = best_matrix.columns.intersection(cols)
    recovered_area = len(recovered_rows) * len(recovered_cols)

    print(recovered_area/(complete.shape[0] * complete.shape[1]))

0.001 12
6 4
False
0.2
0.001 22
6 4
False
0.2
0.001 32
6 4
False
0.2
0.001 42
6 4
False
0.2
0.01 42
6 4
False
0.2
0.001 52
6 4
False
0.2
0.001 62
6 4
False
0.2
0.001 72
6 4
False
0.2
0.001 82
6 4
False
0.2
0.001 92
6 4
False
0.2
0.0021544346900318843 12
6 4
False
0.2
0.0021544346900318843 22
6 4
False
0.2
0.0021544346900318843 62
6 4
False
0.2
0.0021544346900318843 32
6 4
False
0.2
0.0021544346900318843 42
6 4
False
0.2
0.0021544346900318843 52
6 4
False
0.2
0.0021544346900318843 82
6 4
False
0.2
0.0021544346900318843 72
6 4
False
0.2
0.0021544346900318843 92
6 4
False
0.2
0.004641588833612777 12
6 4
False
0.2
0.01 52
6 4
False
0.2
0.004641588833612777 22
6 4
False
0.2
0.004641588833612777 32
6 4
False
0.2
0.004641588833612777 42
6 4
False
0.2
0.004641588833612777 62
6 4
False
0.2
0.004641588833612777 52
6 4
False
0.2
0.004641588833612777 72
6 4
False
0.2
0.004641588833612777 82
6 4
False
0.2
0.01 32
6 4
False
0.2
0.004641588833612777 92
6 4
False
0.2
0.01 12
6 4
False
0.2
0.01 22
6 4
